# MOLI and DeepCDR on a small subset: RMSE / Pearson / Spearman

Both repos are used here as the source of data, drug graphs, the triplet selector and the architecture, but **neither original script runs as-is on a modern Colab**:
- DeepCDR needs Keras 2.1.4 / TensorFlow 1.13 / Python 2 (its training script contains a Python 2 `print` statement), so `prog/model.py` and `prog/layers/graph.py` are re-implemented in PyTorch below.
- MOLI's original scripts are **binary classifiers** (R/S) that need the TCGA/PDX files. RMSE, Pearson and Spearman need a continuous target, so MOLI is run here as a **regression** model on ln IC50 (same per-omic encoders and triplet loss; BCE head replaced by an MSE head).

Everything comes from the `DeepCDR.zip` data (CCLE omics + GDSC ln IC50 + drug graphs), so no extra downloads are needed.
Both models use the **same cell-line train/val/test split** and the **same drugs**.

Upload `MOLI.zip` and `DeepCDR.zip` to Drive and edit the paths in the first cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MOLI_ZIP    = '/content/drive/MyDrive/MOLI.zip'      # edit
DEEPCDR_ZIP = '/content/drive/MyDrive/DeepCDR.zip'   # edit
OUT_DIR     = '/content/drive/MyDrive/moli_deepcdr_subset'

import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
def sh(cmd): return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout

sh(f'unzip -q -o "{MOLI_ZIP}" -d /content')
sh(f'unzip -q -o "{DEEPCDR_ZIP}" -d /content')
sh('unzip -q -o /content/DeepCDR/data/GDSC/drug_graph_feat.zip -d /content/DeepCDR/data/GDSC')
sh('pip -q install hickle h5py')
DATA = '/content/DeepCDR/data'
sys.path.insert(0, '/content/MOLI')                  # utils.py: AllTripletSelector (used by MOLI)
print(len(os.listdir(f'{DATA}/GDSC/drug_graph_feat')), 'drug graph files')

In [ ]:
import csv, copy, random
import numpy as np, pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split

SEED = 42
N_LINES, N_DRUGS = 200, 10          # size of the small subset
VAL_FRAC, TEST_FRAC = 0.10, 0.20    # cell-line level split
EPOCHS_DEEPCDR, EPOCHS_MOLI = 30, 40
PATIENCE = 10
MAX_ATOMS = 100
SMOKE = False                       # True: 2 epochs each, just to check nothing crashes
if SMOKE: EPOCHS_DEEPCDR = EPOCHS_MOLI = 2
random.seed(SEED); np.random.seed(SEED)

## 1. Tables: response, omics, sex, drug-to-PubChem map

In [ ]:
ic50 = pd.read_csv(f'{DATA}/CCLE/GDSC_IC50.csv', index_col=0)                       # rows: 'GDSC:<drug id>', cols: cell lines (ACH-)
rows = list(csv.reader(open(f'{DATA}/GDSC/1.Drug_listMon Jun 24 09_00_55 2019.csv')))
drugid2cid = {r[0]: r[5] for r in rows if r[5].isdigit()}                           # GDSC drug id -> PubChem CID

E  = pd.read_csv(f'{DATA}/CCLE/genomic_expression_561celllines_697genes_demap_features.csv', index_col=0)
C  = pd.read_csv(f'{DATA}/CCLE/genomic_copynumber_561celllines_710genes_demap_features.csv', index_col=0)
Me = pd.read_csv(f'{DATA}/CCLE/genomic_methylation_561celllines_808genes_demap_features.csv', index_col=0)
M  = pd.read_csv(f'{DATA}/CCLE/genomic_mutation_34673_demap_features.csv', index_col=0)   # 961 lines; we keep the 561
C  = C.fillna(0)                                                                           # a few lines have missing CN values

ann = pd.read_csv(f'{DATA}/CCLE/Cell_lines_annotations_20181226.txt', sep='\t').drop_duplicates('depMapID').set_index('depMapID')
sex_all = ann['Gender'].where(ann['Gender'].isin(['male', 'female']), 'unknown')

all_lines = [l for l in E.index if l in set(ic50.columns) and l in set(M.index) and l in set(C.index) and l in set(Me.index)]
print(len(all_lines), 'cell lines with all four omics + response')

# long format; the 15 PubChem CIDs reached by two GDSC drug ids are averaged
long = ic50.loc[[i for i in ic50.index if i.split(':')[1] in drugid2cid], all_lines].copy()
long['cid'] = [drugid2cid[i.split(':')[1]] for i in long.index]
long = long.groupby('cid').mean().stack().dropna().rename('y').reset_index().rename(columns={'level_1': 'line'})   # dropna: newer pandas keeps missing cells
print(len(long), 'line-drug pairs')

## 2. Drug graphs
`drug_graph_feat/<PubChemCID>.hkl` holds `[atom_features, adj_list, degree_list]`. Graphs are padded to 100 atoms and the adjacency matrix is symmetric-normalized with self-loops, exactly as `CalculateGraphFeat` does in `run_DeepCDR.py`.
Drugs with more than 100 atoms are dropped (the original script would crash on them).

In [ ]:
def load_hkl(path):
    import hickle as hkl
    return hkl.load(path)           # if this fails on the old hickle-3 files, see the note in the next cell

def normalize_adj(adj):
    if adj.shape[0] == 0: return adj
    adj = adj + np.eye(adj.shape[0])
    D = np.diag(np.power(adj.sum(1), -0.5))
    return adj.dot(D).T.dot(D)

def calc_graph_feat(feat_mat, adj_list, max_atoms=MAX_ATOMS):
    n = feat_mat.shape[0]
    assert n == len(adj_list)
    feat = np.zeros((max_atoms, feat_mat.shape[1]), np.float32); feat[:n] = feat_mat
    adj = np.zeros((max_atoms, max_atoms), np.float32)
    for i, nodes in enumerate(adj_list):
        for j in nodes: adj[i, int(j)] = 1
    assert np.allclose(adj, adj.T)
    adj[:n, :n] = normalize_adj(adj[:n, :n])
    adj[n:, n:] = normalize_adj(adj[n:, n:])      # padded atoms get self-loops only, as in the original
    return feat, adj

In [ ]:
graph_dir = f'{DATA}/GDSC/drug_graph_feat'
graphs, too_big = {}, []
for f in sorted(os.listdir(graph_dir)):
    cid = f.split('.')[0]
    feat_mat, adj_list, _ = load_hkl(f'{graph_dir}/{f}')
    if feat_mat.shape[0] > MAX_ATOMS: too_big.append(cid); continue
    graphs[cid] = calc_graph_feat(np.asarray(feat_mat), adj_list)
print(len(graphs), 'drug graphs loaded;', len(too_big), 'dropped for >', MAX_ATOMS, 'atoms')

## 3. Small subset and cell-line split
The `N_DRUGS` best-covered drugs that have a graph, and `N_LINES` random cell lines with data for most of them.
The split is by **cell line** (stratified on sex), so no test line is seen in training, for either model.

In [ ]:
long = long[long['cid'].isin(graphs)]
cov = long.groupby('cid')['line'].nunique().sort_values(ascending=False)
drugs = list(cov.index[:N_DRUGS])
sub = long[long['cid'].isin(drugs)]
per_line = sub.groupby('line')['cid'].nunique()
eligible = per_line[per_line >= int(0.7 * N_DRUGS)].index.tolist()
rng = np.random.default_rng(SEED)
lines = sorted(rng.choice(eligible, size=min(N_LINES, len(eligible)), replace=False).tolist())
sub = sub[sub['line'].isin(lines)].reset_index(drop=True)

sex = sex_all.reindex(lines)
trval, te_lines = train_test_split(lines, test_size=TEST_FRAC, stratify=sex.loc[lines].values, random_state=SEED)
tr_lines, va_lines = train_test_split(trval, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=sex.loc[trval].values, random_state=SEED)
split_of = {**{l: 'train' for l in tr_lines}, **{l: 'val' for l in va_lines}, **{l: 'test' for l in te_lines}}
sub['split'] = sub['line'].map(split_of)
sub['sex'] = sub['line'].map(sex)

line2i, cid2i = {l: i for i, l in enumerate(lines)}, {c: i for i, c in enumerate(drugs)}
sub['li'], sub['di'] = sub['line'].map(line2i), sub['cid'].map(cid2i)
print(f"{len(lines)} lines, {len(drugs)} drugs, {len(sub)} pairs")
print(sub.groupby('split').agg(pairs=('y', 'size'), lines=('line', 'nunique')))
print('sex of test lines:', sex.loc[te_lines].value_counts().to_dict())

# per-line omics arrays aligned to `lines`
A = {'E': E.loc[lines].values.astype(np.float32), 'C': C.loc[lines].values.astype(np.float32),
     'Me': Me.loc[lines].values.astype(np.float32), 'M': M.loc[lines].values.astype(np.float32)}
drug_feat = np.stack([graphs[c][0] for c in drugs]); drug_adj = np.stack([graphs[c][1] for c in drugs])
tr_idx = [line2i[l] for l in tr_lines]

## 4. Metrics (pooled and per-drug, overall and by sex)

In [ ]:
def _m(y, p):
    if len(y) < 3: return dict(n=len(y), RMSE=np.nan, Pearson=np.nan, Spearman=np.nan)
    return dict(n=len(y), RMSE=float(np.sqrt(np.mean((p - y) ** 2))),
                Pearson=float(pearsonr(p, y)[0]), Spearman=float(spearmanr(p, y)[0]))

def summarize(pred_df, name):
    out = []
    for grp, d in [('all', pred_df), ('female', pred_df[pred_df.sex == 'female']), ('male', pred_df[pred_df.sex == 'male'])]:
        r = _m(d['y'].values, d['pred'].values)
        pd_ = [_m(g['y'].values, g['pred'].values) for _, g in d.groupby('cid') if len(g) >= 5]
        r['per-drug Pearson'] = np.nanmean([x['Pearson'] for x in pd_]) if pd_ else np.nan
        r['per-drug Spearman'] = np.nanmean([x['Spearman'] for x in pd_]) if pd_ else np.nan
        out.append({'model': name, 'group': grp, **r})
    return pd.DataFrame(out)

## 5. DeepCDR (PyTorch port of `prog/model.py` + `layers/graph.py`)
Drug branch: 4 graph-conv layers (256, 256, 256, 100; ReLU, BatchNorm, dropout 0.1) with global average pooling (the repo's default, `GAP`).
Omics branches: mutation CNN, expression MLP, methylation MLP (100-d each). Concatenate, dense 300, 1-D conv head, linear output.
Adam lr 1e-3, batch 64, MSE; early stopping on **validation** Pearson (the original script uses the test set for this, which leaks).
Copy number is not used by DeepCDR, as in the original.

In [ ]:
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
T = lambda a: torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)

class GraphConv(nn.Module):
    # h' = A^T (X W + b), A already normalized (graph.py: batch_dot(permute(edges), features @ W + b))
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o)
        nn.init.xavier_uniform_(self.lin.weight); nn.init.zeros_(self.lin.bias)
    def forward(self, x, adj): return torch.bmm(adj.transpose(1, 2), self.lin(x))

class SeqBN(nn.Module):      # Keras BatchNormalization(axis=-1) on (batch, atoms, units)
    def __init__(self, u): super().__init__(); self.bn = nn.BatchNorm1d(u, eps=1e-3)
    def forward(self, x): b, n, u = x.shape; return self.bn(x.reshape(b * n, u)).reshape(b, n, u)

class DeepCDR(nn.Module):
    def __init__(self, drug_dim, mut_dim, gexp_dim, methy_dim, units=(256, 256, 256), use_gmp=False):
        super().__init__()
        dims = [drug_dim, *units, 100]
        self.convs = nn.ModuleList([GraphConv(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.bns = nn.ModuleList([SeqBN(b) for b in dims[1:]])
        self.use_gmp, self.drop = use_gmp, nn.Dropout(0.1)
        self.mut = nn.Sequential(nn.Conv1d(1, 50, 700, stride=5), nn.Tanh(), nn.MaxPool1d(5),
                                 nn.Conv1d(50, 30, 5, stride=2), nn.ReLU(), nn.MaxPool1d(10), nn.Flatten())
        with torch.no_grad(): flat = self.mut(torch.zeros(1, 1, mut_dim)).shape[1]
        self.mut_fc = nn.Sequential(nn.Linear(flat, 100), nn.ReLU(), nn.Dropout(0.1))
        mlp = lambda d: nn.Sequential(nn.Linear(d, 256), nn.Tanh(), nn.BatchNorm1d(256, eps=1e-3), nn.Dropout(0.1),
                                      nn.Linear(256, 100), nn.ReLU())
        self.gexp, self.methy = mlp(gexp_dim), mlp(methy_dim)
        self.fuse = nn.Sequential(nn.Linear(400, 300), nn.Tanh(), nn.Dropout(0.1))
        self.head = nn.Sequential(nn.Conv1d(1, 30, 150), nn.ReLU(), nn.MaxPool1d(2),
                                  nn.Conv1d(30, 10, 5), nn.ReLU(), nn.MaxPool1d(3),
                                  nn.Conv1d(10, 5, 5), nn.ReLU(), nn.MaxPool1d(3),
                                  nn.Dropout(0.1), nn.Flatten(), nn.Dropout(0.2))
        with torch.no_grad(): hf = self.head(torch.zeros(1, 1, 300)).shape[1]
        self.out = nn.Linear(hf, 1)
    def forward(self, feat, adj, mut, gexp, methy):
        h = feat
        for conv, bn in zip(self.convs, self.bns): h = self.drop(bn(torch.relu(conv(h, adj))))
        x_drug = h.max(1)[0] if self.use_gmp else h.mean(1)      # padded atoms are pooled too, as in the original
        x_mut = self.mut_fc(self.mut(mut.unsqueeze(1)))
        x = self.fuse(torch.cat([x_drug, x_mut, self.gexp(gexp), self.methy(methy)], dim=1))
        return self.out(self.head(x.unsqueeze(1))).squeeze(-1)

tE, tMe, tM = T(A['E']), T(A['Me']), T(A['M'])
dF, dA = T(drug_feat), T(drug_adj)

def batch_inputs(li, di):
    li, di = torch.as_tensor(li, device=dev), torch.as_tensor(di, device=dev)
    return dF[di], dA[di], tM[li], tE[li], tMe[li]

def predict_deepcdr(model, d, bs=256):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(d), bs):
            s = d.iloc[i:i + bs]; out.append(model(*batch_inputs(s.li.values, s.di.values)).cpu().numpy())
    return np.concatenate(out)

def train_deepcdr(epochs, bs=64):
    torch.manual_seed(SEED)
    trn, val = sub[sub.split == 'train'].reset_index(drop=True), sub[sub.split == 'val'].reset_index(drop=True)
    model = DeepCDR(drug_feat.shape[-1], A['M'].shape[1], A['E'].shape[1], A['Me'].shape[1]).to(dev)
    opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.MSELoss()
    y_tr = T(trn.y.values); best, best_state, wait = -np.inf, None, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(trn)); tot = 0
        for i in range(0, len(perm) - bs + 1, bs):              # drop last partial batch (BatchNorm)
            b = perm[i:i + bs].numpy()
            loss = lossf(model(*batch_inputs(trn.li.values[b], trn.di.values[b])), y_tr[b])
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        pv = predict_deepcdr(model, val); pcc = pearsonr(pv, val.y.values)[0]
        pcc = -1 if np.isnan(pcc) else pcc
        print(f"epoch {ep:3d}  train loss {tot:8.2f}  val Pearson {pcc:.3f}")
        if pcc > best: best, wait, best_state = pcc, 0, copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= PATIENCE: print('early stop'); break
    model.load_state_dict(best_state); return model

In [ ]:
dc_model = train_deepcdr(EPOCHS_DEEPCDR)
test = sub[sub.split == 'test'].copy()
test['pred'] = predict_deepcdr(dc_model, test)
test.assign(model='DeepCDR').to_csv(f'{OUT_DIR}/test_preds_deepcdr.csv', index=False)
res_dc = summarize(test, 'DeepCDR'); res_dc

## 6. MOLI as a regression model (one model per drug)
Same structure as the repo's late-fusion script: one encoder per omic (Linear -> BatchNorm -> ReLU -> Dropout), concatenated, then a head.
Adapted: expression, mutation and copy number (MOLI's three omics); the head predicts ln IC50 with MSE; the triplet loss from the repo's `utils.AllTripletSelector` is kept, using sensitive/resistant = below/above the training median ln IC50 for that drug.
No L2 normalization of the fused embedding (that was for the classifier). Adam instead of Adagrad.
Expression and copy number are standardized with training-line statistics.

In [ ]:
from utils import AllTripletSelector
from sklearn.preprocessing import StandardScaler
trip_sel = AllTripletSelector()

class MOLIReg(nn.Module):
    def __init__(self, in_dims, h_dims=(128, 128, 128), rate=0.3):
        super().__init__()
        self.encs = nn.ModuleList([nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(rate))
                                   for d, h in zip(in_dims, h_dims)])
        self.head = nn.Linear(sum(h_dims), 1)
    def forward(self, xs):
        z = torch.cat([e(x) for e, x in zip(self.encs, xs)], dim=1)
        return self.head(z).squeeze(-1), z

scE = StandardScaler().fit(A['E'][tr_idx]); scC = StandardScaler().fit(A['C'][tr_idx])
X_moli = [T(scE.transform(A['E'])), T(A['M']), T(scC.transform(A['C']))]

def predict_moli(model, li):
    model.eval()
    with torch.no_grad(): return model([x[torch.as_tensor(li, device=dev)] for x in X_moli])[0].cpu().numpy()

def train_moli_drug(d_tr, d_va, epochs, bs=32, lam=0.1, margin=1.0):
    model = MOLIReg([x.shape[1] for x in X_moli]).to(dev)
    opt, mse = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4), nn.MSELoss()
    trip = nn.TripletMarginLoss(margin=margin, p=2)
    y_tr = T(d_tr.y.values); lab = (y_tr < y_tr.median()).float()
    bs = min(bs, max(4, len(d_tr) // 2)); best, best_state = np.inf, None
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(d_tr))
        for i in range(0, len(perm) - bs + 1, bs):
            b = perm[i:i + bs]
            pred, z = model([x[torch.as_tensor(d_tr.li.values[b.numpy()], device=dev)] for x in X_moli])
            loss = mse(pred, y_tr[b])
            t = trip_sel.get_triplets(z, lab[b])
            if t.dim() == 2 and len(t) > 0: loss = loss + lam * trip(z[t[:, 0]], z[t[:, 1]], z[t[:, 2]])
            opt.zero_grad(); loss.backward(); opt.step()
        v = float(np.sqrt(np.mean((predict_moli(model, d_va.li.values) - d_va.y.values) ** 2)))
        if v < best: best, best_state = v, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); return model, best

In [ ]:
torch.manual_seed(SEED); moli_rows = []
for cid in drugs:
    d = sub[sub.cid == cid]
    d_tr, d_va, d_te = d[d.split == 'train'], d[d.split == 'val'], d[d.split == 'test']
    if len(d_tr) < 20 or len(d_va) < 3 or len(d_te) < 3: print('skip', cid, '(too few lines)'); continue
    model, vr = train_moli_drug(d_tr, d_va, EPOCHS_MOLI)
    moli_rows.append(d_te.assign(pred=predict_moli(model, d_te.li.values)))
    print(f"drug {cid}: train={len(d_tr)} val RMSE={vr:.3f}")
moli_test = pd.concat(moli_rows)
moli_test.assign(model='MOLI (regression)').to_csv(f'{OUT_DIR}/test_preds_moli.csv', index=False)
res_moli = summarize(moli_test, 'MOLI (regression)'); res_moli

## 7. Side by side

In [ ]:
summary = pd.concat([res_moli, res_dc], ignore_index=True)
summary.round(3).to_csv(f'{OUT_DIR}/summary_metrics.csv', index=False)
print(summary.round(3).to_string(index=False))